# Customer Churn Prediction (IBM Telco)\n
This notebook uses the real Kaggle Telco dataset and follows the project instructions.

In [ ]:
import warnings\n
warnings.filterwarnings('ignore')\n
\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
\n
from sklearn.compose import ColumnTransformer\n
from sklearn.impute import SimpleImputer\n
from sklearn.model_selection import train_test_split\n
from sklearn.pipeline import Pipeline\n
from sklearn.preprocessing import OneHotEncoder, StandardScaler\n
from sklearn.linear_model import LogisticRegression\n
from sklearn.ensemble import RandomForestClassifier\n
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve\n
\n
sns.set_theme(style='whitegrid')

In [ ]:
data_path = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'\n
df = pd.read_csv(data_path)\n
df.head()

In [ ]:
print('Shape:', df.shape)\n
display(df.dtypes)\n
display(df.isnull().sum())\n
print('Churn rate:', round((df['Churn'].eq('Yes').mean() * 100), 2), '%')

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')\n
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [ ]:
plt.figure(figsize=(6,4))\n
sns.countplot(data=df, x='Churn')\n
plt.title('Churn Distribution')\n
plt.show()

In [ ]:
contract_rate = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)\n
plt.figure(figsize=(8,4))\n
sns.barplot(x=contract_rate.index, y=contract_rate.values)\n
plt.title('Churn Rate by Contract')\n
plt.ylabel('Churn Rate')\n
plt.xticks(rotation=20)\n
plt.show()

In [ ]:
df_model = df.drop(columns=['customerID']).copy()\n
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})\n
\n
X = df_model.drop(columns=['Churn'])\n
y = df_model['Churn']\n
\n
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']\n
categorical_cols = [c for c in X.columns if c not in numeric_cols]\n
\n
preprocessor = ColumnTransformer([\n
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),\n
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),\n
])\n
\n
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
models = {\n
    'logistic_regression': Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))]),\n
    'random_forest': Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1))])\n
}\n
\n
results = []\n
for name, model in models.items():\n
    model.fit(X_train, y_train)\n
    pred = model.predict(X_test)\n
    prob = model.predict_proba(X_test)[:, 1]\n
    results.append({\n
        'model': name,\n
        'accuracy': accuracy_score(y_test, pred),\n
        'precision': precision_score(y_test, pred),\n
        'recall': recall_score(y_test, pred),\n
        'f1': f1_score(y_test, pred),\n
        'roc_auc': roc_auc_score(y_test, prob)\n
    })\n
\n
pd.DataFrame(results).sort_values(['f1', 'roc_auc'], ascending=False)

In [ ]:
best = models['random_forest']\n
best.fit(X_train, y_train)\n
pred = best.predict(X_test)\n
prob = best.predict_proba(X_test)[:, 1]\n
\n
cm = confusion_matrix(y_test, pred)\n
plt.figure(figsize=(5,4))\n
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')\n
plt.title('Confusion Matrix - Random Forest')\n
plt.xlabel('Predicted')\n
plt.ylabel('Actual')\n
plt.show()\n
\n
fpr, tpr, _ = roc_curve(y_test, prob)\n
plt.figure(figsize=(6,4))\n
plt.plot(fpr, tpr, label='Random Forest')\n
plt.plot([0,1],[0,1],'k--')\n
plt.xlabel('FPR')\n
plt.ylabel('TPR')\n
plt.title('ROC Curve')\n
plt.legend()\n
plt.show()